# El mecanismo de Attention

## Introducción
Attention es un mecanismo complejo de abordar al principio. Hay mucha data dando vueltas que, lamentablemente no clarifica nada si no al contrario. A no preocuparse si los conceptos tardan en decantar y a no desesperarse. La mejor fuente para entender el mecanismo de manera visual es la lista de Neural Networks del canal de Youtube 3Blue1Brown (three blue, one brown), en particular los videos 5, 6 y 7, que tratan de modelos de lenguajes, Attention y Transformers, son de particular valor para el contenido de este curso. Si bien los videos están en inglés, la traducción automática es semi-descente.

### Donde estamos parados y hacia donde vamos?

![transformer_diagram.png](attachment:117c4050-6995-415c-9340-dfae99e76e6e.png)

![attention_diagram.png](attachment:b700d9c3-1289-405a-b84f-dd747bd8aa1a.png)

Vamos a tratar de abordar Attention con el mismo enfoque que Andrej Karpathy en sus videos, es decir, como un mecanismo de transmisión de información a través del modelo. La información se transmite desde los embeddings del contexto de un token al embedding que lo representa.

[LLM Visualization](https://bbycroft.net/llm)


### Que esperamos del mecanismo de Attention?
En el modelo de transformers, los embeddings de los tokens deben modificarse para asimilar información de su contexto. Por ejemplo:
![e_tower.png](attachment:366c4af8-42e1-4464-a5ce-ac42110bdc9a.png)

![e_Eiffel_tower.png](attachment:5d9701ad-7f38-41ef-80f4-856ce0c970fb.png)

![e_miniature_Eiffel_tower.png](attachment:a808f114-13ad-41e3-946c-c91004fc8254.png)

¿Qué información se transmite? Depende. Depende del contexto en el cual el token está inmerso, depende de los datos de entrenamiento, depende de lo que el modelo aprendió de esos datos.

1) El embedding de un token es determinístico. Una vez que el modelo está entrenado, es siempre el mismo.
2) El transformer (decoder) actualiza el embedding de todos los tokens del contexto, usando información de todos los tokens del contexto anterior, para predecir el siguiente token
3) La probabilidad del token siguiente al último del contexto, depende exclusivamente del embedding (modificado, después de "fluir" a través del modelo) del último token del contexto.

![tokens_embeddings.png](attachment:cab2ac21-a9cf-4faa-83fc-514aec324058.png)

![prediction_next_token.png](attachment:c95c1053-210a-47b0-b03d-b46bf3d6f4ee.png)

![fluffy_blue_creature_example.png](attachment:e788f369-c20b-4f08-8f69-f78dcbb32b77.png)

El código para implementar un Attention Head es sorprendentemente simple. Ahora entender a fondo que hace y porqué es tan efectivo es otra cosa.

In [ ]:
import torch.nn as nn
class AttentionHead(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)  #la x son los embeddings
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C **-.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

### Un ejemplo mínimo de Dot Product Attention
Vamos a ver un ejemplo muy chico las operaciones que haría un attention head y vamos a ver las diferencias entre los diferentes mecanismos de attention.

In [ ]:
import torch
import torch.nn.functional as F
from matplotlib import pyplot as plt
T = 8  # cantidad de tokens
C = 5  # cantidad de dimensiones de los embeddings que representan cada token
torch.manual_seed(42)
emb = torch.randn(T, C) #matriz de 8x5

In [ ]:
emb

In [ ]:
#información a transmitir va a ser el promedio entre tokens 
att = torch.zeros(T, C)
for t in range(T):
    embprev = emb[:t+1] #a cada token se le sum lo que tiene el anterior 
    att[t] = torch.mean(embprev,0)  # la información comunicada es el promedio de los embeddings del contexto
att  #el primeor no se modifica, la segunda fila de att es la suma de las primeras dos filas de T divididos por 2. 

In [ ]:
lim=3
emb[:lim],torch.mean(emb[:lim],dim=0)

In [ ]:
(1.9269+ -1.2345)/2

### Haciendo lo mismo pero multiplicando matrices
Se puede hacer lo mismo y ser muy muy MUY eficiente haciendo la misma operación pero multiplicando matrices. La ventaja es que la multiplicación de matrices es una operación masivamente paralela. Ténicamente, el tipo de paralelismo de esta operación es SIMD (Single Instruction Multiple Data), que significa que se hace la misma operación (sumas y multiplicaciones) pero en múltiples datos a la vez. Por lo tanto, se pueden usar placas de video para hacer estas operaciones como si los datos de las matrices fuera pixels, (kind of).

In [ ]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a /= a.sum(1, keepdims=True)
b = torch.randint(0,10,(3,2)).float()
c = a@b

In [ ]:
print('a')
print(a , '3x3')
print('b')
print(b, '3x2')
print('c')
print(a@b,'3x3 @ 3x2 --> 3x2')

In [ ]:
wei = torch.tril(torch.ones(T, T))
wei /= wei.sum(1, keepdims=True)

In [ ]:
wei

In [ ]:
att2 = wei @ emb
att2, att

### Agregando Softmax
Una forma más flexible de construir la matriz `wei` es usando softmax. Enseguida vamos a ver por que esto es mejor...

In [ ]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))  #esta máscara impide que los tokens viajan a los futuros. 
wei = wei, dim=-1
att3 = wei @ emb  #es la misma matriz de peso que la anterior pero hecho con softmax

In [ ]:
att, att2, att3;

In [ ]:
F.softmax(torch.zeros(T, T).masked_fill(tril == 0, float('-inf')),dim=1)

De esta manera `wei` puede interpretarse como una matriz que representa la afinidad de los tokens entre si. Al comienzo se inicializa en ceros y a medida que avanza el entrenamiento, `wei` aprende que tokens son más interesantes para uno en particular.

### Agregando la dimensión del batch
Para entrenar el mecanismo de attention hace falta una cantidad importante de datos. Por lo tanto no vamos a mandar los ejemplos de a poquito, vamos a mandarlos en batches. Por suerte, todo lo que hemos hecho hasta ahora y el poder de broadcasting de arrays de pytorch nos va a dejar agregar la dimensión de batch relativamente fácil.

Lo interesante aqui es que cada batch se procesa independientemente. Un ejercicio interesante es tratar de imaginarse el grafo de operaciones de `wei@emb`. Este grafo de operaciones todavía puede ser ampliado para incluir el cálculo de una función de pérdida y sobre el cual se puede hacer `.backwards()` para actualizar los pesos de `wei`. 

In [ ]:
B = 4
T = 8
C = 5
torch.manual_seed(42)
emb = torch.randn(B, T, C)
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
att = wei@emb

In [ ]:
# (4, 8, 8) @ (4,8,5) > (4,8,5)  #broadcasting de arrays: cuando tengo que multiplicar wei que es de 8x8 por los subs que es de 8x5
# (1, 8, 8)
# (4, 8, 5)
emb.shape, wei.shape, att.shape

In [ ]:
att[1], emb[1]

### El Mecanismo de Attention
El promedio no es una forma interesante de comunicación entre tokens. El mecanismo de attention calcula dos vectores a partir de los embeddings de un token $E_i$. Un vector $Q_i$ es la `query`, indica que es lo que el token con embedding $E_i$ está buscando. El segundo vector es $K_i$, o `key`, y representa de alguna manera lo que el token **es**. La dimensión de estos vectores $Q_i$ y $K_i$ generalmente es menor que la dimensión del embedding. En el mecanismo de Attetion interviene un tercer vector llamado $V_i$ de values. Este vector es la respuesta a como debe modificarse el embedding $E_i$ en el contexto de los demás tokens. Pero hacer esto para un solo token/embedding no tiene sentido. Es necesarion hacerlo para todos los tokens/embeddings del contexto a la vez.

En nuestro ejemplo, si la dimensión del batch es `B`, el tamaño del contexto es `T` y la dimensión de los embeddings es `C`, para calcular $Q_i$ y $K_i$ debemos crear dos matrices de dimensión `(C, head_dimension)` donde `head_dimension` es una variable que hace referencia a la cantidad dimensiones de $Q_i$ y $K_i$ (y consecuentemente, $V_i$). Lo que vamos a hacer entonces en implementar en Python la fórmula del paper ***Attention is all you need***.

$Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

```python
head_dimension = 2
B = 4
T = 8
C = 5
tril = torch.tril(torch.ones(T,T))  # Esto es un buffer
Wq = torch.randn(C, head_dimension) # (5,2 )
Wk = torch.randn(C, head_dimension)
Wv = torch.randn(C, head_dimension)
emb = torch.randn(B, T, C)
Q = emb @ Wq     # (4, 8,5) @ (5, 2) --> (4, 8, 2)
K = emb @ Wk
wei = Q @ K.T * C ** -.5    # (4, 8, 2) @ (4, 2, 8) --> (4, 8, 8)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
V = emb @ Wv # (4, 8, 2)
att = wei @ V  #(4, 8, 8) @ (4, 8, 2) --> (4, 8, 2)
```

Usamos producto punto para determinar que tan bien alineados están los vectores $Q_i$ con $K_i$. Recordemos que

![vector_dot_product1.png](attachment:049012d3-2f40-4727-ad87-5e8ee55d1a46.png)
![vector_dot_product2.png](attachment:98f1f9df-fdd8-452b-a78c-0d4c7a3d1fef.png)

Si $Q_i$ y $K_i$ están muy alineados, son muy afines, entonces el softmax va a devolver valores altos. Si por el contrario, $K_i$ no es importante para $Q_i$, entonces el softmax va a ser cercano a cero.

In [ ]:
torch.manual_seed(42)
head_dimension = 2
B = 4
T = 8
C = 5
tril = torch.tril(torch.ones(T,T))  # Esto es un buffer
Wq = torch.randn(C, head_dimension)
Wk = torch.randn(C, head_dimension)
Wv = torch.randn(C, head_dimension)
emb = torch.randn(B, T, C)
Q = emb @ Wq
K = emb @ Wk
wei = Q @ K.transpose(-2, -1) * C ** -.5  # Por esto se llama "scaled" dot product attention
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
V = emb @ Wv
att = wei @ V

In [ ]:
att.shape

In [ ]:
plt.imshow(wei[0])

## Notas importantes
* Attention puede verse como un mecanismo de comunicación entre nodos en un grafo dirigido. En el caso de Masked Self Attention, el grafo tiene un a estructura particular.
![attention_graph.png](attachment:dffc34a1-af1a-46ff-86de-43151dd16aaa.png)

* No hay una noción de espacio como en el caso de las redes convolucionales. Por eso es necesario tener un embedding para el token sino también un embedding para la posición del token (possitional embedding).
* No hay comunicacion entre batches. Los batches se procesan independientemente de los demás. Esto se hace para aprovechar el paralelismo que proveen las GPUs.


* Cross Attention vs Self Attention
* Masked Attention vs Unmasked Attention
* Multi-head Attention vs Single Head Attention
* Encoder block vs Decoder Block
  
![transformer_diagram.png](attachment:3c8e5484-e277-44f9-ac8d-85dbf4943617.png)

* Scaled Attention. La escala es importante por que, sobre todo durante la inicialización, si `wei` no es difusa, el `softmax(wei)` va a converger a algo de tipo one hot. Es decir, vamos a estar agregando información solamente de un nodo, y la información de los demás nodos va a desaparecer.

In [ ]:
BERT 

### Recursos
* [Attention is all you need paper](https://arxiv.org/abs/1706.03762v1)
* [3Blue1Brown - Neural Networks series](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi)
* [LLM Visualization](https://bbycroft.net/llm)
* [Christian Olah blog](https://colah.github.io/)

## Resumen
* El mecanismo de atención como herramienta de trasmisión de información
  * Trasmitiendo información a un token desde el contexto
  * Transformando embeddings
* Scaled dot-product Attention
* Tipos de Attention
  * Self Attention vs Cross Attention
  * Masked vs Unmasked Attention (causal)
  * Single Head vs Multi Head Attention
* Implementación de los diferentes mecanismos de atención

## Que sigue?

In [ ]:
a = torch.randn((3,3,3,5,4))
print(a, a.shape)
b = torch.randn((4,4))
print(b, b.shape)
a@b, (a@b).shape